In [1]:
! pip install pdfplumber python-docx


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


<h3> Resume Parsing </h3>

In [21]:
import os
import re
import pdfplumber
from docx import Document


class ResumeParser:
    def __init__(self, static_folder="static"):
        self.static_folder = static_folder

    # -------------------------
    # Public Method
    # -------------------------
    def parse_resume(self, filename: str) -> dict:
        """
        Main function to parse resume from /static directory.
        Returns structured dictionary.
        """

        file_path = os.path.join(self.static_folder, filename)

        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Resume not found at {file_path}")

        text = self._extract_text(file_path)

        if not text or len(text.strip()) < 100:
            raise ValueError("Resume text extraction failed or document is empty.")

        sections = self._split_sections(text)

        return {
            "profile_header": self._parse_profile_header(sections.get("PROFILE", "")),
            "education": self._parse_education(sections.get("EDUCATION", "")),
            "work_experience": self._parse_work_experience(sections.get("WORK EXPERIENCE", "")),
            "projects": self._parse_projects(sections.get("PROJECTS", "")),
        }

    # -------------------------
    # Text Extraction
    # -------------------------
    def _extract_text(self, file_path: str) -> str:
        try:
            if file_path.lower().endswith(".pdf"):
                return self._extract_pdf(file_path)
            elif file_path.lower().endswith(".docx"):
                return self._extract_docx(file_path)
            else:
                raise ValueError("Unsupported file format. Only PDF and DOCX allowed.")
        except Exception as e:
            raise RuntimeError(f"Error extracting text: {str(e)}")

    def _extract_pdf(self, file_path: str) -> str:
        text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text

    def _extract_docx(self, file_path: str) -> str:
        doc = Document(file_path)
        return "\n".join([para.text for para in doc.paragraphs])

    # -------------------------
    # Section Splitter
    # -------------------------
    def _split_sections(self, text: str) -> dict:
        section_titles = [
            "PROFILE",
            "EDUCATION",
            "WORK EXPERIENCE",
            "PROJECTS",
        ]

        pattern = r"(?=\n?(PROFILE|EDUCATION|WORK EXPERIENCE|PROJECTS)\n)"
        splits = re.split(pattern, text)

        sections = {}
        current_title = None

        for chunk in splits:
            chunk = chunk.strip()
            if chunk in section_titles:
                current_title = chunk
                sections[current_title] = ""
            elif current_title:
                sections[current_title] += chunk.strip() + "\n"

        return sections

    # -------------------------
    # Profile Header
    # -------------------------
    def _parse_profile_header(self, profile_text: str) -> dict:
        return {
            "summary": profile_text.strip()
        }

    # -------------------------
    # Education Parser
    # -------------------------
    
    def _parse_education(self, education_text: str) -> list:
        education_entries = []

        if not education_text.strip():
            return education_entries

        entries = re.split(r"\n(?=[A-Z][A-Za-z\s&]+ - )", education_text.strip())

        for entry in entries:
            entry = entry.strip()
            if not entry:
                continue


            # UNIVERSITY NAME

            university_match = re.match(r"^(.*?) -", entry)
            if not university_match:
                continue

            university = university_match.group(1).strip()

 
            # DATE 
         
            date_pattern = r"([A-Za-z]{3,9}\s+\d{4})(?:\s*(?:-|to)\s*([A-Za-z]{3,9}\s+\d{4}|Present|present))?"
            date_match = re.search(date_pattern, entry)

            start_date = None
            end_date = "present"

            if date_match:
                start_date = date_match.group(1).strip()
                if date_match.group(2):
                    end_date = date_match.group(2).strip()

           
            # DEGREE FULL TEXT
            # Remove university and date portion
            degree_text = entry

            # Remove university part
            degree_text = re.sub(rf"^{re.escape(university)}\s*-\s*", "", degree_text)

            # Remove date portion
            if date_match:
                degree_text = degree_text.replace(date_match.group(0), "").strip()

            # Remove bullet points
            degree_text = degree_text.split("•")[0].strip()

           
            degree = None
            major = None

            # Common degree keywords
            degree_keywords = [
                "Bachelor of",
                "Master of",
                "B.Tech",
                "M.Tech",
                "BSc",
                "MSc",
                "PhD",
                "Doctor of"
            ]

            for keyword in degree_keywords:
                if degree_text.startswith(keyword):
                    parts = degree_text.split(" ", 3)

                    # Example:
                    # Bachelor of Technology Electronics and Communications Engineering
                    # We assume first 3 words = degree
                    degree_parts = degree_text.split()
                    if len(degree_parts) >= 3:
                        degree = " ".join(degree_parts[:3])
                        major = " ".join(degree_parts[3:]) if len(degree_parts) > 3 else None
                    else:
                        degree = degree_text
                    break

            # Fallback if pattern not matched
            if not degree:
                words = degree_text.split()
                if len(words) > 3:
                    degree = " ".join(words[:3])
                    major = " ".join(words[3:])
                else:
                    degree = degree_text

            

            education_entries.append({
                "university": university,
                "degree": degree.strip() if degree else None,
                "major": major.strip() if major else None,
                "start_date": start_date,
                "end_date": end_date,
            })

        return education_entries

    # -------------------------
    # Work Experience Parser
    # -------------------------
    def _parse_work_experience(self, work_text: str) -> list:
        experiences = []

        entries = re.split(r"\n(?=[A-Za-z\s]+ \| )", work_text)

        for entry in entries:
            entry = entry.strip()
            if not entry:
                continue

            header_match = re.match(r"(.*?) \| (.*?) ([A-Za-z]+\s\d{4}\s(?:to|-)\s[A-Za-z]+\s\d{4})", entry)

            if header_match:
                position = header_match.group(1).strip()
                company = header_match.group(2).strip()
                
                # dates = header_match.group(3).strip()
                
                # DATE 
         
                date_pattern = r"([A-Za-z]{3,9}\s+\d{4})(?:\s*(?:-|to)\s*([A-Za-z]{3,9}\s+\d{4}|Present|present))?"
                date_match = re.search(date_pattern, entry)

                start_date = None
                end_date = "present"

                if date_match:
                    start_date = date_match.group(1).strip()
                    if date_match.group(2):
                        end_date = date_match.group(2).strip()

                description = entry[header_match.end():].strip()

                experiences.append({
                    "company": company,
                    "position": position,
                    "start_date": start_date,
                    "end_date": end_date,
                    "description": description
                })

        return experiences

    # -------------------------
    # Projects Parser
    # -------------------------
    def _parse_projects(self, projects_text: str) -> list:
        projects = []

        entries = re.split(r"\n(?=[A-Z][A-Za-z0-9 &]+(?:\n|$))", projects_text)

        for entry in entries:
            lines = entry.strip().split("\n")
            if len(lines) < 2:
                continue

            title = lines[0].strip()
            description = "\n".join(lines[1:]).strip()

            projects.append({
                "project_title": title,
                "project_description": description
            })

        return projects

In [25]:
parser = ResumeParser(static_folder="static")

data = parser.parse_resume("Resume Full Time ML V260.pdf")

print(data["education"])
print(data["work_experience"])
print(data["projects"])



CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


[{'project_title': 'Human Activity Recognition using Limited Training Data', 'project_description': '• Developed strong expertise in multivariate wearable sensor time-series analysis, including multi-\nresolution feature extraction across temporal scales, subject-level variability handling, and human-\nparticipant data considerations.\n• Designed and implemented zero shot learning, few shot learning and transfer learning models on human\nactivity recognition using PAMAP2 dataset enabling systematic comparison in scenarios with limited\ntraining data.\n• Implemented prototype-based few-shot learning and fine-tuned transfer learning models, demonstrating\nsubstantial performance gains over zero-shot approaches on unseen activities.\n• Conducted rigorous, leakage-free evaluation using subject-disjoint splits, confusion matrices, and collapse\ndiagnostics, highlighting real-world deployment challenges in wearable sensor ML.'}, {'project_title': 'EMNIST dataset image classification', 'proje

<h3> Creating Embedding </h3>

In [26]:
! pip install sentence-transformers scikit-learn numpy


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [27]:
# Core libraries
import json
import os
import numpy as np

# Embeddings
from sentence_transformers import SentenceTransformer

# Similarity
from sklearn.metrics.pairwise import cosine_similarity

Build Resume Text

In [29]:
def build_resume_text(resume):
    """
    Convert parsed resume dictionary (based on our parser structure)
    into semantic-rich text for embedding.
    """

    # Profile Summary
    profile_text = resume.get("profile_header", {}).get("summary", "")

    # Education
    education_text = ""
    for edu in resume.get("education", []):
        degree = edu.get("degree", "")
        major = edu.get("major", "")
        university = edu.get("university", "")
        education_text += f"{degree} in {major} from {university}. "

    # Work Experience
    experience_text = ""
    for exp in resume.get("work_experience", []):
        position = exp.get("position", "")
        company = exp.get("company", "")
        description = exp.get("description", "")
        experience_text += f"{position} at {company}. {description} "

    # Projects
    projects_text = ""
    for proj in resume.get("projects", []):
        title = proj.get("project_title", "")
        description = proj.get("project_description", "")
        projects_text += f"{title}: {description} "

    resume_text = f"""
    Candidate Profile:
    {profile_text}

    Education:
    {education_text}

    Work Experience:
    {experience_text}

    Projects:
    {projects_text}
    """

    return resume_text.strip()

In [32]:
resume_text = build_resume_text(data)

In [33]:
# Load job dataset
jobs_path = "data/jobs.json"

with open(jobs_path, "r") as f:
    jobs = json.load(f)

print(f"Total jobs loaded: {len(jobs)}")

Total jobs loaded: 8


In [35]:
def build_job_text(job):
    """
    Convert job JSON into semantic text block.
    """

    skills = ", ".join(job.get("skills_required", []))
    responsibilities = " ".join(job.get("responsibilities", []))

    job_text = f"""
    Job Title: {job.get('title', '')}
    Company: {job.get('company', '')}
    Location: {job.get('location', '')}
    Experience Required: {job.get('experience_required', '')}
    Education Required: {job.get('education_required', '')}
    Required Skills: {skills}
    Job Description: {job.get('job_description', '')}
    Responsibilities: {responsibilities}
    """

    return job_text.strip()


job_texts = [build_job_text(job) for job in jobs]

print(job_texts[0][:500])  # preview first job

Job Title: Machine Learning Engineer
    Company: InnovaAI Solutions
    Location: London, UK
    Experience Required: 2-4 years
    Education Required: Bachelor's or Master's in Computer Science, Data Science, or related field
    Required Skills: Python, PyTorch, TensorFlow, Transformers, NLP, Docker, AWS, MLflow
    Job Description: We are seeking a Machine Learning Engineer to design, build, and deploy scalable ML systems. You will work on transformer-based NLP models, fine-tune large langua


In [36]:
# Load embedding model (free + local)
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embedding model loaded successfully.


In [37]:
# Generate resume embedding
resume_embedding = model.encode(resume_text)

# Generate job embeddings
job_embeddings = model.encode(job_texts)

print("Embeddings generated.")
print("Resume embedding shape:", resume_embedding.shape)
print("Job embeddings shape:", job_embeddings.shape)

Embeddings generated.
Resume embedding shape: (384,)
Job embeddings shape: (8, 384)


In [40]:
# Compute cosine similarity
similarities = cosine_similarity(
    [resume_embedding],
    job_embeddings
)[0]  # extract row

# Attach similarity scores to jobs
for i, job in enumerate(jobs):
    job["semantic_score"] = float(similarities[i])

# Sort jobs by similarity descending
ranked_jobs = sorted(jobs, key=lambda x: x["semantic_score"], reverse=True)

# Display ranked jobs (realistic job-style format)
for i, job in enumerate(ranked_jobs):
    print(f"Title: {job.get('title', 'N/A')}")
    print(f"Company: {job.get('company', 'N/A')}")
    print(f"Location: {job.get('location', 'N/A')}")
    print(f"Experience Required: {job.get('experience_required', 'N/A')}")
    print(f"Semantic Match Score: {job['semantic_score']:.4f}")
    print("\nJob Description:")

    # Clean preview of description (avoid overwhelming notebook)
    description = job.get("job_description", "")
    preview = description[:800] + ("..." if len(description) > 800 else "")
    print(preview)

    print("=" * 100)
    print("\n")

Title: Machine Learning Engineer
Company: InnovaAI Solutions
Location: London, UK
Experience Required: 2-4 years
Semantic Match Score: 0.7679

Job Description:
We are seeking a Machine Learning Engineer to design, build, and deploy scalable ML systems. You will work on transformer-based NLP models, fine-tune large language models, and develop end-to-end data pipelines. Responsibilities include model training, hyperparameter tuning, deployment using Docker, and monitoring using MLflow. Experience with AWS and MLOps best practices such as CI/CD and model versioning is highly desirable.


Title: MLOps Engineer
Company: CloudScale AI
Location: Remote
Experience Required: 2-5 years
Semantic Match Score: 0.6638

Job Description:
We are looking for an MLOps Engineer to streamline the deployment and monitoring of machine learning models. You will implement CI/CD pipelines, manage Kubernetes clusters, and ensure scalable deployment on AWS. Experience with Apache Airflow and model monitoring too

In [51]:
top_5_jobs = ranked_jobs[:5]

print("\nTop 5 Jobs After Semantic Filtering:\n")

for job in top_5_jobs:
    print(f"{job['title']} | Score: {job['semantic_score']:.4f}")


Top 5 Jobs After Semantic Filtering:

Machine Learning Engineer | Score: 0.7679
MLOps Engineer | Score: 0.6638
Data Scientist | Score: 0.5773
NLP Engineer | Score: 0.5596
AI Research Engineer | Score: 0.5467


<h3> LLM Reranking and prompt based reasoning </h3>

In [41]:
! pip install langchain langchain-groq python-dotenv

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


INFO: pip is looking at multiple versions of langchain-groq to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 2.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.7/604.7 kB 3.3 MB/s  0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.14.0
    Uninstalling typing_extensions-4.14.0:
      Successfully uninstalled typing_extensions-4.14.0━━━━━━━━━━━  1/10 [typing-extensions]
  Attempting uninstall: packaging━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/10 [typing-extensions]
    Found existing installation: packaging 25.0━━━━━━━━━━━━━━━  1/10 [typing-extensions]
    Uninstalling packaging-25.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/10 [typing-extensions]
      Successfully uninstalled packaging-25.0━━━━━━━━━━━━━━━━━  1/10 [typing-extensions]
  Attempting uninstall: typing-inspection━━━━━━━━━━━━━━━━━━━━━  1/10 [typing-extensions]
    Fo

In [53]:
resume_text

"Candidate Profile:\n    PROFILE\nI have 2.5+ years of experience as a machine learning engineer with specialisation in building and deploying\nproduction-grade ML and Generative AI systems, with a strong focus on transformer-based models, LLM\nfine-tuning, NLP pipelines, and MLOps. Experienced in end-to-end system development, applied machine\nlearning, data pipelines, and user-facing analytical tools, with a strong interest in skills intelligence, workforce\nanalytics, and applied research-led innovation.\n\n    Education:\n    Master of Science in Data Science and AI from University of Liverpool. Bachelor of Technology in Electronics and Communications Engineering from Vellore Institute of Technology. \n\n    Work Experience:\n    Machine Learning and Data Engineer at Valuechain Technology Ltd. • Automated data extraction using Selenium and built an ETL pipeline using Apache Airflow, ensuring\nefficient data ingestion and transformation. The data was stored using Amazon Redshift.\n•

In [45]:
import os
from dotenv import load_dotenv
import json
from langchain_groq import ChatGroq
from langchain.prompts import ChatPromptTemplate
from langchain.schema import SystemMessage, HumanMessage

In [56]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load .env
load_dotenv()

# Initialize Groq client (reads GROQ_API_KEY from environment)
client = Groq()

print("Groq client initialized.")

Groq client initialized.


In [61]:
# resume_text

In [69]:
import json

def llm_call(resume_text, job):
    """
    Uses Groq openai/gpt-oss-120b model
    to score and analyze a job match.
    """

    system_prompt = """
You are an expert career recommendation AI.

Analyze how well a candidate matches a job posting.

Return STRICT JSON ONLY in this format:
{
  "match_score": number between 0 and 100,
  "reasoning": "short explanation",
  "skill_gaps": ["gap1", "gap2"],
  "improvement_suggestions": ["suggestion1", "suggestion2"]
}

Do NOT include markdown.
Do NOT include extra text.
Return ONLY JSON.
"""

    user_prompt = f"""
CANDIDATE RESUME:
{resume_text}

JOB DETAILS:
Title: {job.get('title')}
Company: {job.get('company')}
Location: {job.get('location')}
Experience Required: {job.get('experience_required')}
Education Required: {job.get('education_required')}
Required Skills: {", ".join(job.get('skills_required', []))}
Job Description: {job.get('job_description')}
Responsibilities: {" ".join(job.get('responsibilities', []))}

Evaluate match.
"""

    completion = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.3,
        max_completion_tokens=2048,
        top_p=1,
        reasoning_effort="medium",
        stream=True
    )

    full_response = ""

    for chunk in completion:
        content = chunk.choices[0].delta.content
        if content:
            full_response += content

    # Try parsing JSON safely
    try:
        parsed = json.loads(full_response)
    except:
        print("⚠ JSON parsing failed. Raw output returned.")
        parsed = {
            "match_score": 0,
            "reasoning": full_response,
            "skill_gaps": [],
            "improvement_suggestions": []
        }

    return parsed

In [70]:
# resume_text = build_resume_text(resume_data)

final_ranked_jobs = []

for job in top_5_jobs:
    print(f"\nAnalyzing: {job['title']} at {job['company']}")

    analysis = llm_call(resume_text, job)

    job["llm_analysis"] = analysis
    job["llm_score"] = analysis.get("match_score", 0)

    final_ranked_jobs.append(job)

# Sort by LLM score
final_ranked_jobs = sorted(
    final_ranked_jobs,
    key=lambda x: x["llm_score"],
    reverse=True
)

print("\n🔥 FINAL RANKING AFTER LLM RE-RANKING\n")

for i, job in enumerate(final_ranked_jobs):
    print("="*100)
    print(f"RANK {i+1}")
    print(f"Title: {job['title']}")
    print(f"Company: {job['company']}")
    print(f"LLM Match Score: {job['llm_score']}")
    print("\nReasoning:")
    print(job["llm_analysis"]["reasoning"])
    print("\nSkill Gaps:")
    print(job["llm_analysis"]["skill_gaps"])


Analyzing: Machine Learning Engineer at InnovaAI Solutions

Analyzing: MLOps Engineer at CloudScale AI

Analyzing: Data Scientist at Quantiva Analytics

Analyzing: NLP Engineer at TextIQ Systems

Analyzing: AI Research Engineer at DeepVision Labs

🔥 FINAL RANKING AFTER LLM RE-RANKING

RANK 1
Title: Machine Learning Engineer
Company: InnovaAI Solutions
LLM Match Score: 90

Reasoning:
The candidate has 2.5+ years of ML engineering experience, strong expertise in transformer-based NLP models, LLM fine-tuning, MLOps, Docker, AWS, and MLflow—all core requirements. Their skill set aligns closely with the job's technical stack and responsibilities.

Skill Gaps:
['Limited explicit experience with large-scale production monitoring tools beyond MLflow/Prometheus', 'No mention of hands‑on work with TensorFlow serving or SageMaker for model deployment']
RANK 2
Title: MLOps Engineer
Company: CloudScale AI
LLM Match Score: 90

Reasoning:
The candidate has 2.5+ years of relevant ML engineering exper

<h3> Gen AI based career assistant </h3>

In [71]:
required_vars = ["data", "resume_text", "final_ranked_jobs"]

missing = [var for var in required_vars if var not in globals()]

if missing:
    raise ValueError(f"Missing required variables: {missing}. Run ranking pipeline first.")
    
if not final_ranked_jobs:
    raise ValueError("final_ranked_jobs is empty.")

print("Required data validated successfully.")

Required data validated successfully.


In [78]:
len(final_ranked_jobs)

5

In [72]:
# ============================================================
# CELL 2: Robust LLM Call Utility
# Handles streaming + safe JSON parsing
# ============================================================

import json

def call_llm(system_prompt, user_prompt, expect_json=False):
    """
    Generic LLM wrapper for controlled responses.
    """

    try:
        completion = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.3,
            max_completion_tokens=2048,
            top_p=1,
            reasoning_effort="medium",
            stream=True
        )

        full_response = ""

        for chunk in completion:
            content = chunk.choices[0].delta.content
            if content:
                full_response += content

        if expect_json:
            try:
                return json.loads(full_response)
            except:
                print("⚠ JSON parsing failed. Returning raw output.")
                return {"raw_output": full_response}

        return full_response

    except Exception as e:
        print(f"LLM call failed: {e}")
        return None

In [ ]:
# full_response = ""

#     for chunk in completion:
#         content = chunk.choices[0].delta.content
#         if content:
#             full_response += content

#     # Try parsing JSON safely
#     try:
#         parsed = json.loads(full_response)
#     except:
#         print("⚠ JSON parsing failed. Raw output returned.")
#         parsed = {
#             "match_score": 0,
#             "reasoning": full_response,
#             "skill_gaps": [],
#             "improvement_suggestions": []
        }

In [ ]:
# ============================================================
# CELL 3 (UPDATED): Job-Specific Skill Gap + Roadmap
# ============================================================

def identify_skill_gaps_for_job(job_index):
    """
    Shows skill gaps and generates improvement roadmap
    for a specific selected job.
    """

    # --- Validation ---
    if job_index < 1 or job_index > len(final_ranked_jobs):
        print("❌ Invalid job index.")
        return

    selected_job = final_ranked_jobs[job_index - 1]
    job_title = selected_job.get("title")
    company = selected_job.get("company")

    skill_gaps = selected_job.get("llm_analysis", {}).get("skill_gaps", [])

    print(f"\n🎯 Skill Gap Analysis for: {job_title} at {company}\n")

    if not skill_gaps:
        print("✅ No major skill gaps identified.")
        return

    # --- Display Skill Gaps ---
    print("📌 Missing / Weak Skills:")
    for skill in skill_gaps:
        print(f"- {skill}")

    # --- Generate Roadmap ---
    system_prompt = """
You are a senior career advisor AI.

Given missing skills for a specific job, create:

1. Priority order of skills to learn
2. Practical learning roadmap (3-6 months)
3. Suggested project ideas to strengthen profile
4. Industry-relevant advice

Be structured and practical.
"""

    user_prompt = f"""
Candidate Resume:
{resume_text}

Selected Job:
Title: {job_title}
Company: {company}
Description: {selected_job.get('job_description')}
Required Skills: {selected_job.get('skills_required')}

Identified Skill Gaps:
{skill_gaps}

Create a focused improvement roadmap for THIS job only.
"""

    roadmap = call_llm(system_prompt, user_prompt)

    print("\n🛣 Improvement Roadmap:\n")
    print(roadmap)

In [79]:
final_ranked_jobs

[{'job_id': 'J001',
  'title': 'Machine Learning Engineer',
  'company': 'InnovaAI Solutions',
  'location': 'London, UK',
  'employment_type': 'Full-time',
  'experience_required': '2-4 years',
  'education_required': "Bachelor's or Master's in Computer Science, Data Science, or related field",
  'skills_required': ['Python',
   'PyTorch',
   'TensorFlow',
   'Transformers',
   'NLP',
   'Docker',
   'AWS',
   'MLflow'],
  'job_description': 'We are seeking a Machine Learning Engineer to design, build, and deploy scalable ML systems. You will work on transformer-based NLP models, fine-tune large language models, and develop end-to-end data pipelines. Responsibilities include model training, hyperparameter tuning, deployment using Docker, and monitoring using MLflow. Experience with AWS and MLOps best practices such as CI/CD and model versioning is highly desirable.',
  'responsibilities': ['Develop and deploy ML models in production',
   'Fine-tune transformer-based NLP models',
   'D

In [74]:
# ============================================================
# CELL 4: Explain Matching Quality Across Jobs
# ============================================================

def explain_matching_quality(top_jobs):

    summary = []

    for i, job in enumerate(top_jobs):
        summary.append({
            "rank": i+1,
            "title": job.get("title"),
            "company": job.get("company"),
            "semantic_score": job.get("semantic_score"),
            "llm_score": job.get("llm_score"),
            "reasoning": job.get("llm_analysis", {}).get("reasoning")
        })

    system_prompt = """
You are a career analytics AI.
Explain how well the candidate matches these jobs.
Highlight patterns in strengths and weaknesses.
Be analytical and clear.
"""

    user_prompt = f"""
Candidate Resume:
{resume_text}

Ranked Job Summary:
{summary}

Explain:
1. Why top job ranks highest
2. Why lower ranked jobs rank lower
3. Overall pattern of fit
"""

    explanation = call_llm(system_prompt, user_prompt)

    print("\n📈 Matching Explanation:\n")
    print(explanation)

In [75]:
# ============================================================
# CELL 5: Show Similar Work Done for Selected Job
# ============================================================

def summarize_similar_experience(job_index):

    if job_index < 1 or job_index > len(final_ranked_jobs):
        print("❌ Invalid job index.")
        return

    selected_job = final_ranked_jobs[job_index - 1]

    system_prompt = """
You are a resume alignment expert.
Identify past work or projects that align closely with the job.
Quote specific experiences and summarize clearly.
"""

    user_prompt = f"""
Candidate Resume:
{resume_text}

Selected Job:
Title: {selected_job.get('title')}
Company: {selected_job.get('company')}
Description: {selected_job.get('job_description')}
Required Skills: {selected_job.get('skills_required')}

Identify:
1. Most relevant work experience
2. Most relevant projects
3. Why they align
"""

    response = call_llm(system_prompt, user_prompt)

    print(f"\n🔎 Similar Work for: {selected_job.get('title')} at {selected_job.get('company')}\n")
    print(response)

In [76]:
# ============================================================
# CELL 6: Manual Selection Control
# ============================================================

# Set manually:
user_choice = 1  # 1, 2, or 3

if user_choice == 1:
    identify_skill_gaps(final_ranked_jobs)

elif user_choice == 2:
    explain_matching_quality(final_ranked_jobs)

elif user_choice == 3:
    selected_job_number = 1  # Change this manually
    summarize_similar_experience(selected_job_number)

else:
    print("Invalid choice. Select 1, 2, or 3.")

📊 Aggregated Skill Gaps (Frequency Based):
Limited explicit experience with large-scale production monitoring tools beyond MLflow/Prometheus → 1 times
No mention of hands‑on work with TensorFlow serving or SageMaker for model deployment → 1 times
Experience with dedicated model monitoring tools such as Grafana, Seldon Core, or Evidently AI → 1 times
Hands‑on management of large‑scale production Kubernetes clusters (e.g., EKS) and autoscaling → 1 times
Deep familiarity with advanced AWS MLOps services like SageMaker or AWS CodePipeline → 1 times
Advanced statistical inference / hypothesis testing → 1 times
Formal experience presenting insights to business stakeholders → 1 times
Experience with industry-standard BI tools like Tableau → 1 times
Explicit BERT fine‑tuning experience not highlighted → 1 times
Advanced inference optimization (quantization, distillation, latency tuning) → 1 times
Computer vision experience → 1 times
Research track record / publications → 1 times
Large‑scale mo